In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import glob
import pandas as pd
from datetime import datetime
from sunpy.coordinates import sun

In [2]:
df = pd.read_csv('wcs.csv', skipinitialspace=True).dropna()

dids = df['did'].to_numpy()
dates = np.array([datetime.fromisoformat(date) for date in df['date']])
sec = np.array([(datetime.fromisoformat(date) - dates[0]).total_seconds() for date in df['date']])

car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()
crln = df['crln_obs'].to_numpy()
crlt = df['crlt_obs'].to_numpy()
dsun = df['dsun_au'].to_numpy()
fgh_tsp = df['fgh_tsp'].to_numpy()
contposn = df['contposn'].to_numpy()
obs_vw = df['obs_vw'].to_numpy() #/ 1000
obs_vn = df['obs_vn'].to_numpy() #/ 1000
obs_vr = df['obs_vr'].to_numpy() #/ 1000

b0 = np.array(sun.B0(dates))

x, y, z = np.cos(crlt * np.pi / 180) * np.cos(hgln * np.pi / 180), np.cos(crlt * np.pi / 180) * np.sin(hgln * np.pi / 180), np.sin(crlt * np.pi / 180)
xe, ye, ze = np.cos(b0 * np.pi / 180), 0, np.sin(b0 * np.pi / 180)
phi = np.arccos(x * xe + y * ye + z * ze) * 180 / np.pi

In [3]:
plt.rcParams['date.converter'] = 'concise'

t = np.where((car_rot[1:] - car_rot[:-1]) != 0)[0] + 1

fig, ax = plt.subplots(figsize=(10,8))
ax_ = ax.twinx()
ax1 = ax.twiny()


for a, b in zip(dates[t[::2]], dates[t[1::2]]):
    ax.axvspan(a, b, color='tab:blue', alpha=0.1)

#ax.plot(dates, dsun, color='black', lw=1.25)
ax.plot(dates, crln, color='black', lw=1.25)
#ax.plot(dates, hgln, color='black', lw=1.25)
ax_.plot(dates, crlt, '.', color='tab:red', lw=1.25)

#ax.set_xlim(datetime(2026,1,1), datetime(2026,9,1))
#ax.set_ylim(0.2,1.)
ax_.set_ylim(-20,20)

ax_.tick_params(axis='y', colors='tab:red')

ax1.set_xlim(ax.get_xlim())
ax1.set_xticks(dates[t])
ax1.set_xticklabels(car_rot[t], rotation='vertical', size='small')

ax.set_xlabel('Date')
ax.set_ylabel('Carrington longitude, degrees')
ax_.set_ylabel('Carrington latitude, degrees')
ax1.set_xlabel('Carrington rotation number')

ax.grid(True, axis='y')
plt.tight_layout()

In [4]:
plt.rcParams['date.converter'] = 'concise'

t = np.where((car_rot[1:] - car_rot[:-1]) != 0)[0] + 1

fig, ax = plt.subplots(figsize=(10,8))
ax_ = ax.twinx()
ax1 = ax.twiny()


for a, b in zip(dates[t[::2]], dates[t[1::2]]):
    ax.axvspan(a, b, color='tab:blue', alpha=0.1)

ax.plot(dates, hgln, color='black', lw=1.25)
ax.plot(dates, phi, '--', color='black', lw=1.25)
ax_.plot(dates, crlt - b0, color='tab:red', lw=1.25)
ax_.plot(dates, b0, '--', color='tab:red', lw=1.25)
#ax_.plot(dates, b0, color='tab:red', lw=1.25)

#ax.set_xlim(datetime(2023,12,31), datetime(2026,1,1))
ax.set_ylim(-200,200)
ax_.set_ylim(-40,40)

ax_.tick_params(axis='y', colors='tab:red')

ax1.set_xlim(ax.get_xlim())
ax1.set_xticks(dates[t])
ax1.set_xticklabels(car_rot[t], rotation='vertical', size='small')

ax.set_xlabel('Date')
ax.set_ylabel('Stonyhurst longitude, degrees')
ax_.set_ylabel(r'$\Delta B_0$, degrees')
ax1.set_xlabel('Carrington rotation number')
#plt.title('SOLO-Earth Separation angles')

ax.grid(True, axis='y')
plt.tight_layout()

In [38]:
from transforms import *

WSID = 360 / 25.38
AU = 149597870691.

crln_ = crln + WSID * sec / 24 / 3600

rot = ~Rotate.x(crlt) + Rotate.y(crln_)
r = rot((0, 0, 1))[0]
q = rot((-obs_vn / dsun / AU * 24 * 3600 * 180 / np.pi, obs_vw / dsun / AU * 24 * 3600 * 180 / np.pi, 0))[0]

fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(projection='3d', aspect='equal')
ax.scatter(*r)
ax.scatter(*q)


ax.set_xlim(-1,1)
ax.set_ylim(-1,1)
ax.set_zlim(-1,1)

(-1.0, 1.0)

In [39]:
#i = -195
#i = -240
i = -200

w = obs_vw / dsun / AU * 24 * 3600 * 180 / np.pi / np.cos(crlt * np.pi / 180)
#w = q[1]

-(((crln[i] - crln[i + 10]) - 180) % 360 - 180) / ((w[i] + w[i + 10]) / 2 - WSID), (sec[i + 10] - sec[i]) / 24 / 3600, crlt[i], crlt[i + 10]

(np.float64(7.292173682569554),
 np.float64(7.340277777777778),
 np.float64(-4.62286),
 np.float64(8.561412))

In [40]:
def dot(x, y):
    return x[0] * y[0] + x[1] * y[1] + x[2] * y[2]

def cross(x, y):
    return x[1] * y[2] - x[2] * y[1], x[2] * y[0] - x[0] * y[2], x[0] * y[1] - x[1] * y[0]

def norm(x):
    return np.sqrt(sum(x_ ** 2 for x_ in x))

In [41]:
r_ = np.array(r).T
q_ = np.array(q).T

#np.arccos(dot(r_[i], r_[i + 20])) * 180 / np.pi / norm((q_[i] + q_[i + 1]) / 2), (sec[i + 20] - sec[i]) / 24 / 3600

np.arctan2(dot(cross(r_[i], r_[i + 10]), q_[i]) / norm(q_[i]), dot(r_[i], r_[i + 10])) / norm((q_[i] + q_[i + 10]) / 2) * 180 / np.pi, (sec[i + 10] - sec[i]) / 24 / 3600

(np.float64(7.457335616108156), np.float64(7.340277777777778))

In [43]:

Q

np.float64(0.8212982302538809)

In [44]:
alpha = 1.327124e20
AU = 149597870691.

Vt = np.sqrt(obs_vw ** 2 + obs_vn ** 2)
Vr = obs_vr

r = dsun * AU
p = (r * Vt) ** 2 / alpha

V0 = np.sqrt(alpha / p)
e = np.sqrt((Vt / V0 - 1) ** 2 + (Vr / V0) ** 2)
a = p / (1 - e ** 2)

cos_f = (Vt / V0 - 1) / e
sin_f = (Vr / V0) / e

cosE = (e + cos_f) / (1 + e * cos_f)
sinE = np.sqrt(1 - e ** 2) * sin_f / (1 + e * cos_f)

theta = np.arctan2(sin_f, cos_f)
E = np.arctan2(sinE, cosE)

t = a * np.sqrt(a / alpha) * (E - e * sinE)

In [69]:
i = 500

Q = np.arctan2(dot(cross(r_[i], r_[i + 10]), q_[i]) / norm(q_[i]), dot(r_[i], r_[i + 10]))

theta1 = theta[i] + Q
sin_f1 = np.sin(theta1)
cos_f1 = np.cos(theta1)

cosE1 = (e[i] + cos_f1) / (1 + e[i] * cos_f1)
sinE1 = np.sqrt(1 - e[i] ** 2) * sin_f1 / (1 + e[i] * cos_f1)

E1 = np.arctan2(sinE1, cosE1)
t1 = a[i] * np.sqrt(a[i] / alpha) * (E1 - e[i] * sinE1)

(t1 - t[i]) / 24 / 3600, (sec[i + 10] - sec[i]) / 24 / 3600

(np.float64(1.2501345757886884), np.float64(1.25))

In [65]:
plt.figure(figsize=(10,10))
plt.plot(dates, t)